# 20 - Superpose pockets per ligand (for ChimeraX)

For **every** ligand code with >=2 binding structures, write one PDB
per structure (ligand + encoded pocket), all superposed on the ligand
so they overlay in ChimeraX. Output is PDBs only.

Per code:
- gather all binding structures across the four datasets;
- **one structure per PDB id** (we want different PDBs, not repeated
  chains of the same entry);
- if more than `MAX_STRUCTURES` (100) remain, keep a diverse 100 by
  **round-robin over UniProt id** (cached `pdb_to_uniprot.json` maps),
  so the set spans as many distinct proteins as possible;
- ligand = heavy atoms from `DATA/<ds>/raw/<entry>/<ligand_file>`
  (altlocs resolved, H dropped; CCD atom names kept for matching);
- pocket = `pocket_atoms`/`pocket_coordinates` from `<ds>_pocket.lmdb`;
- reference = the most-complete ligand; every other structure is
  rigid-fit onto it by its ligand (Kabsch / `Bio.SVDSuperimposer`,
  shared atom names), and the transform is applied to its pocket.

Writes `RESULTS/pocket_viz/<CODE>/<CODE>__NN__<pocket>.pdb`
(ligand chain L, pocket chain P) and one `RESULTS/pocket_viz/_index.tsv`.
In ChimeraX: `open RESULTS/pocket_viz/<CODE>/*.pdb` loads them as
separate, already-aligned models.

A final section plots, per ligand, the distribution of pairwise
**pocket-embedding** cosine similarities (inline only, nothing saved).

Regenerate via `scripts/build_visualise_pockets_notebook.py`.

## Setup

In [10]:
import json, pickle, warnings
from pathlib import Path

import lmdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt   # only used by the optional helper
from tqdm.auto import tqdm
from Bio.PDB import PDBParser
from Bio.PDB.Atom import DisorderedAtom
from Bio.PDB.Residue import DisorderedResidue
from Bio.PDB.PDBExceptions import PDBConstructionWarning
from Bio.SVDSuperimposer import SVDSuperimposer
warnings.simplefilter('ignore', PDBConstructionWarning)

PROJECT = Path.cwd().parent / 'DATA' / 'natural_ligands'
DATA_ROOT = PROJECT / 'DATA'
POCKETS_DIR = PROJECT / 'RESULTS' / 'pockets'
COMPARE_DIR = PROJECT / 'RESULTS' / 'pocket_compare'
OUT_ROOT = PROJECT / 'RESULTS' / 'pocket_viz'
OUT_ROOT.mkdir(parents=True, exist_ok=True)

# ---- config ----
MAX_STRUCTURES = 1000000     # cap per ligand code
MIN_OCCURRENCES = 2      # only codes seen at least this many times
MIN_COMMON_ATOMS = 3     # need >=3 shared ligand atoms to align
REF_INDEX = None         # None -> most-complete ligand; or pin an int
ONLY_CODES = ["NAP"]        # e.g. ['NAD','FAD'] to limit; None = all
# ----------------

DATASETS = {'coach420': 'coach420', 'scpdb': 'sc-pdb',
            'pdbbind2020': 'pdbbind2020', 'holo4k': 'holo4k'}
WATER = {'HOH', 'WAT', 'DOD', 'H2O', 'TIP', 'TIP3', 'SOL'}
PARSER = PDBParser(QUIET=True)

PDB2UNI = {}
for rep in DATASETS:
    p = COMPARE_DIR / rep / 'pdb_to_uniprot.json'
    if p.exists():
        for k, v in json.load(open(p)).items():
            PDB2UNI.setdefault(k.lower(), v)
def pdb_uniprot(pdb):
    v = PDB2UNI.get(str(pdb).lower())
    return v[0] if v else ''
print(f'PDB->UniProt entries: {len(PDB2UNI):,}')

PDB->UniProt entries: 12,651


## Index + selection

Concatenate the four pocket manifests, group by ligand code, and pick
the structures to keep per code (one per PDB; UniProt round-robin when
over the cap).

In [11]:
frames = []
for rep, man in DATASETS.items():
    mpath = POCKETS_DIR / f'{man}_pocket_manifest.tsv'
    if not mpath.exists():
        continue
    df = pd.read_csv(mpath, sep='\t', dtype=str)
    df['rep'] = rep
    frames.append(df)
index = pd.concat(frames, ignore_index=True)
by_code = {c: d for c, d in index.groupby('lig_resname')}
codes = sorted(c for c, d in by_code.items()
               if len(d) >= MIN_OCCURRENCES)
if ONLY_CODES is not None:
    codes = [c for c in codes if c in set(ONLY_CODES)]
print(f'{len(index):,} structures; {len(codes):,} ligand codes with '
      f'>= {MIN_OCCURRENCES} occurrences')

def select_structures(rows):
    '''One row per PDB id; if > MAX_STRUCTURES, round-robin over UniProt
    to keep a protein-diverse subset.'''
    reps = rows.sort_values('pocket').drop_duplicates('pdb_id',
                                                      keep='first')
    if len(reps) <= MAX_STRUCTURES:
        return reps
    groups = {}
    for _, r in reps.iterrows():
        u = pdb_uniprot(r['pdb_id']) or ('_pdb_' + str(r['pdb_id']))
        groups.setdefault(u, []).append(r)
    keys = list(groups.keys())
    picked = []
    while len(picked) < MAX_STRUCTURES:
        progressed = False
        for k in keys:
            if groups[k]:
                picked.append(groups[k].pop(0))
                progressed = True
                if len(picked) >= MAX_STRUCTURES:
                    break
        if not progressed:
            break
    return pd.DataFrame(picked)

17,305 structures; 1 ligand codes with >= 2 occurrences


## Helpers

Altloc/element/name normalisation mirror notebook 16. One pocket-lmdb
env is cached per dataset (path is `<rep>_pocket.lmdb` or
`<rep>/pocket.lmdb`).

In [12]:
def _occ(a):
    o = a.get_occupancy()
    return 0.0 if o is None else float(o)

def resolve_altlocs(structure):
    for chain in structure.get_chains():
        for residue in list(chain):
            if isinstance(residue, DisorderedResidue):
                ids = residue.disordered_get_id_list()
                best = max(ids, key=lambda r: sum(
                    _occ(a) for a in
                    residue.disordered_get(r).get_unpacked_list()))
                residue.disordered_select(best)
                res = residue.disordered_get(best)
            else:
                res = residue
            for atom in list(res):
                if isinstance(atom, DisorderedAtom):
                    alt = max(atom.disordered_get_id_list(),
                              key=lambda a: _occ(atom.child_dict[a]))
                    atom.disordered_select(alt)

def norm_element(atom):
    el = (atom.element or '').strip()
    if not el:
        nm = atom.get_name().strip()
        el = ''.join(c for c in nm if c.isalpha())[:2]
    return el.capitalize() if len(el) == 2 else el.upper()

def norm_name(n):
    return n.strip().replace(chr(42), chr(39))   # '*' -> prime

def read_ligand(pdb_path):
    s = PARSER.get_structure('lig', str(pdb_path))
    resolve_altlocs(s)
    names, elems, coords, seen = [], [], [], set()
    for res in s.get_residues():
        if res.get_resname().strip().upper() in WATER:
            continue
        for a in res.get_unpacked_list():
            el = norm_element(a)
            if el in ('H', 'D'):
                continue
            nm = norm_name(a.get_name())
            if nm in seen:
                continue
            seen.add(nm)
            names.append(nm)
            elems.append(el)
            coords.append(np.asarray(a.coord, np.float32))
    c = np.stack(coords) if coords else np.zeros((0, 3), np.float32)
    return names, elems, c.astype(np.float32)

_ENV = {}
def read_pocket(rep, lmdb_key):
    if rep not in _ENV:
        cands = [POCKETS_DIR / f'{rep}_pocket.lmdb',
                 POCKETS_DIR / rep / 'pocket.lmdb']
        p = next((c for c in cands if c.exists()), None)
        if p is None:
            raise FileNotFoundError(f'no pocket lmdb for {rep}')
        _ENV[rep] = lmdb.open(str(p), subdir=False, readonly=True,
                              lock=False, readahead=False)
    with _ENV[rep].begin() as t:
        rec = pickle.loads(t.get(str(lmdb_key).encode()))
    coords = np.asarray(rec['pocket_coordinates'], np.float32).reshape(-1, 3)
    return list(rec['pocket_atoms']), coords

def superpose(ref_named, mov_named):
    common = [n for n in ref_named if n in mov_named]
    if len(common) < MIN_COMMON_ATOMS:
        return None
    rc = np.stack([ref_named[n] for n in common]).astype(np.float64)
    mc = np.stack([mov_named[n] for n in common]).astype(np.float64)
    sup = SVDSuperimposer()
    sup.set(rc, mc)
    sup.run()
    rot, tran = sup.get_rotran()
    return rot, tran, float(sup.get_rms()), len(common)

def apply_xf(coords, rot, tran):
    return (coords.astype(np.float64) @ rot + tran).astype(np.float32)

## Align + write functions

`align_code` returns the superposed records (no I/O); `write_code`
writes the PDBs. Keeping them separate makes ad-hoc exploration easy
(call `align_code` then `plot_overlay`).

In [13]:
def align_code(code, rows):
    recs = []
    for _, m in rows.iterrows():
        lig_pdb = (DATA_ROOT / m['dataset'] / 'raw' / m['entry']
                   / m['ligand_file'])
        try:
            if not lig_pdb.exists():
                raise FileNotFoundError(lig_pdb)
            names, elems, coords = read_ligand(lig_pdb)
            if len(names) < MIN_COMMON_ATOMS:
                raise ValueError('too few ligand atoms')
            p_atoms, p_coords = read_pocket(m['rep'], m['lmdb_key'])
        except Exception:
            continue
        recs.append({'pocket': m['pocket'], 'rep': m['rep'],
                     'dataset': m['dataset'], 'entry': m['entry'],
                     'pdb_id': m['pdb_id'],
                     'uniprot': pdb_uniprot(m['pdb_id']),
                     'lig_names': names, 'lig_elems': elems,
                     'lig_coords': coords, 'poc_atoms': p_atoms,
                     'poc_coords': p_coords})
    if not recs:
        return []
    if REF_INDEX is not None and REF_INDEX < len(recs):
        ref_i = REF_INDEX
    else:
        ref_i = max(range(len(recs)),
                    key=lambda k: len(recs[k]['lig_names']))
    ref_named = dict(zip(recs[ref_i]['lig_names'],
                         recs[ref_i]['lig_coords']))
    for k, r in enumerate(recs):
        if k == ref_i:
            r['rmsd'], r['n_common'] = 0.0, len(r['lig_names'])
            continue
        out = superpose(ref_named,
                        dict(zip(r['lig_names'], r['lig_coords'])))
        if out is None:
            r['rmsd'], r['n_common'] = float('nan'), 0
            continue
        rot, tran, rms, nc = out
        r['lig_coords'] = apply_xf(r['lig_coords'], rot, tran)
        r['poc_coords'] = apply_xf(r['poc_coords'], rot, tran)
        r['rmsd'], r['n_common'] = rms, nc
    return [r for r in recs if r['n_common'] >= MIN_COMMON_ATOMS]

def _atom_line(serial, name, resn, chain, resseq, xyz, elem):
    return ('HETATM%5d %-4s %-3s %1s%4d    '
            '%8.3f%8.3f%8.3f  1.00  0.00          %2s'
            % (serial, name[:4], resn[:3], chain[:1], resseq,
               float(xyz[0]), float(xyz[1]), float(xyz[2]), elem[:2]))

def _structure_lines(code, r):
    lines, serial = [], 1
    for nm, el, xyz in zip(r['lig_names'], r['lig_elems'],
                           r['lig_coords']):
        lines.append(_atom_line(serial, nm, code, 'L', 999, xyz, el))
        serial += 1
    for el, xyz in zip(r['poc_atoms'], r['poc_coords']):
        lines.append(_atom_line(serial, el, 'POC', 'P', 1, xyz, el))
        serial += 1
    return lines

def write_code(code, aligned):
    out = OUT_ROOT / code
    out.mkdir(parents=True, exist_ok=True)
    for old in out.glob('*'):
        if old.is_file():
            old.unlink()
    rows = []
    for rank, r in enumerate(aligned):
        pk = r['pocket']
        fname = f'{code}__{rank:02d}__{pk}.pdb'
        remark = (f'REMARK  {pk} pdb={r["pdb_id"]} '
                  f'uniprot={r["uniprot"]} rmsd={r["rmsd"]:.3f}')
        body = _structure_lines(code, r)
        (out / fname).write_text(remark + '\n' + '\n'.join(body)
                                 + '\nEND\n')
        rows.append({'code': code, 'rank': rank, 'pocket': pk,
                     'pdb_id': r['pdb_id'], 'uniprot': r['uniprot'],
                     'dataset': r['dataset'],
                     'n_lig_atoms': len(r['lig_names']),
                     'n_common': r['n_common'],
                     'lig_rmsd': round(r['rmsd'], 3),
                     'n_pocket_atoms': len(r['poc_atoms']),
                     'file': str((out / fname).relative_to(PROJECT))})
    return rows

## Run for all ligand codes

Codes that end up with < 2 distinct-PDB structures are skipped (nothing
to compare). Writes one PDB per structure plus `_index.tsv`.

In [14]:
index_rows = []
n_codes = 0
for code in tqdm(codes, desc='ligand codes'):
    rows = select_structures(by_code[code])
    if len(rows) < 2:
        continue
    aligned = align_code(code, rows)
    if len(aligned) < 2:
        continue
    index_rows += write_code(code, aligned)
    n_codes += 1

manifest = pd.DataFrame(index_rows)
manifest.to_csv(OUT_ROOT / '_index.tsv', sep='\t', index=False)
print(f'wrote {len(manifest):,} PDBs across {n_codes:,} ligand codes')
print('index:', OUT_ROOT / '_index.tsv')
manifest.head()

ligand codes: 100%|███████████████████████████████| 1/1 [00:03<00:00,  3.17s/it]

wrote 242 PDBs across 1 ligand codes
index: /Users/jsutges/Documents/NATURAL_LIGANDS/RESULTS/pocket_viz/_index.tsv


,code,rank,pocket,pdb_id,uniprot,dataset,n_lig_atoms,n_common,lig_rmsd,n_pocket_atoms,file
0,NAP,0,coach420_2gz3_NAP_A_367,2gz3,A0A0H2UPS5,coach420,48,48,0.000,286,RESULTS/pocket_viz/NAP/NAP__00__coach420_2gz3_...
1,NAP,1,coach420_2qzz_NAP_A_401,2qzz,Q15GI4,coach420,48,48,2.125,310,RESULTS/pocket_viz/NAP/NAP__01__coach420_2qzz_...
2,NAP,2,coach420_2rk2_NAP_A_1,2rk2,P00383,coach420,48,48,3.062,82,RESULTS/pocket_viz/NAP/NAP__02__coach420_2rk2_...
3,NAP,3,coach420_3baz_NAP_A_500,3baz,Q65CJ7,coach420,48,48,1.953,309,RESULTS/pocket_viz/NAP/NAP__03__coach420_3baz_...
4,NAP,4,coach420_7dfr_NAP_A_164,7dfr,P0ABQ4,coach420,47,47,3.356,275,RESULTS/pocket_viz/NAP/NAP__04__coach420_7dfr_...


In [15]:
manifest.query('code == "ADP"').drop_duplicates(["pocket"])

,code,rank,pocket,pdb_id,uniprot,dataset,n_lig_atoms,n_common,lig_rmsd,n_pocket_atoms,file


## (optional) explore a single code

Nothing is plotted by the run above. To eyeball one ligand's overlay
in the notebook, re-align it and call the helper, e.g.:

```python
recs = align_code('NAD', select_structures(by_code['NAD']))
plot_overlay('NAD', recs)
```

In [9]:
def plot_overlay(code, aligned):
    '''Quick 3D scatter: ligands (orange) overlaid, pockets (blue) cloud.'''
    keep = [r for r in aligned if r['n_common'] >= MIN_COMMON_ATOMS]
    lig = np.concatenate([r['lig_coords'] for r in keep])
    poc = np.concatenate([r['poc_coords'] for r in keep])
    fig = plt.figure(figsize=(6.5, 6))
    ax = fig.add_subplot(111, projection='3d')
    ax.scatter(poc[:, 0], poc[:, 1], poc[:, 2], s=2,
               c='lightsteelblue', alpha=0.2)
    ax.scatter(lig[:, 0], lig[:, 1], lig[:, 2], s=8,
               c='darkorange', alpha=0.5)
    ax.set_title(f'{code}: {len(keep)} pockets superposed on ligand')
    ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
    plt.show()

# example (uncomment to run):
# recs = align_code('AMP', select_structures(by_code['AMP']))
# plot_overlay('AMP', recs)